# CAMA Data Investigation Notebook
## Purpose: Load ZIP, check data quality, verify against pipeline OUT_FIELDS
**Run cells top to bottom. Each cell has a comment explaining what it does.**

In [1]:
# Cell 1 — Imports
import zipfile
import pandas as pd
import numpy as np

ZIP_PATH = 'ACPA_CAMAData.zip'
print('Libraries loaded OK')

Libraries loaded OK


---
## SECTION 1 — Load all source files from ZIP

In [3]:
z = zipfile.ZipFile(ZIP_PATH)

def read(filename):
    with z.open(filename) as f:
        df = pd.read_csv(f, sep='\t', dtype=str, low_memory=False)
    print(f'{filename:<25} {len(df):>10,} rows  |  {len(df.columns)} columns')
    return df

prop  = read('Property.txt')
own   = read('Owners.txt')
hist  = read('HistoryRE.txt')
land  = read('Land.txt')
imprv = read('Improvements.txt')
sales = read('Sales.txt')

Property.txt                 109,360 rows  |  18 columns
Owners.txt                   109,376 rows  |  10 columns
HistoryRE.txt              1,257,542 rows  |  20 columns
Land.txt                     112,652 rows  |  12 columns
Improvements.txt             179,951 rows  |  11 columns
Sales.txt                    503,470 rows  |  11 columns


---
## SECTION 2 — Property.txt investigation

In [4]:
# Cell 3 — DOR_UC (Prop_Use_Code) distribution
# Pipeline cleaner keeps only codes 001, 002, 004, 007 (residential)
# CAMA uses 5-digit codes — load_cama.py takes first 3 chars
# e.g. '00100' → '001'

prop['DOR_norm'] = prop['Prop_Use_Code'].astype(str).str.strip().str[:3]

RESIDENTIAL = ['001', '002', '004', '007']

print('Top 20 DOR codes (first 3 chars):')
top = prop['DOR_norm'].value_counts().head(20)
for code, count in top.items():
    marker = ' ← RESIDENTIAL (kept by cleaner)' if code in RESIDENTIAL else ''
    print(f'  {code}  →  {count:,}{marker}')

res_count = prop[prop['DOR_norm'].isin(RESIDENTIAL)]
print(f'\nTotal parcels       : {len(prop):,}')
print(f'Residential parcels : {len(res_count):,}  ({len(res_count)/len(prop)*100:.1f}%)')
print(f'Non-residential     : {len(prop)-len(res_count):,}  (will be dropped by cleaner)')

Top 20 DOR codes (first 3 chars):
  001  →  65,639 ← RESIDENTIAL (kept by cleaner)
  000  →  9,367
  004  →  7,315 ← RESIDENTIAL (kept by cleaner)
  002  →  5,684 ← RESIDENTIAL (kept by cleaner)
  055  →  2,283
  008  →  1,620
  061  →  1,520
  080  →  1,213
  010  →  1,063
  065  →  968
  007  →  956 ← RESIDENTIAL (kept by cleaner)
  009  →  935
  017  →  786
  048  →  709
  071  →  645
  011  →  639
  094  →  560
  019  →  472
  068  →  444
  093  →  392

Total parcels       : 109,360
Residential parcels : 79,594  (72.8%)
Non-residential     : 29,766  (will be dropped by cleaner)


In [5]:
# Cell 4 — City distribution
print('Cities in ZIP data:')
print(prop['City_Desc'].value_counts().head(20))

Cities in ZIP data:
City_Desc
GAINESVILLE     39085
ST. JOHN'S      29779
SUWANNEE        21072
ALACHUA          6651
NEWBERRY         5110
HIGH SPRINGS     4260
HAWTHORNE        1236
ARCHER            786
WALDO             647
MICANOPY          473
LACROSSE          261
Name: count, dtype: int64


---
## SECTION 3 — HistoryRE.txt investigation

In [6]:
# Cell 5 — Just_Value (JV) quality check
# Pipeline needs JV >= 1000 to pass cleaner Step 2

hist['Just_Value'] = pd.to_numeric(hist['Just_Value'], errors='coerce')
hist['Hist_Tax_Year'] = pd.to_numeric(hist['Hist_Tax_Year'], errors='coerce')

# Get most recent year per parcel (same as load_cama.py does)
hist_latest = hist.sort_values('Hist_Tax_Year', ascending=False)
hist_latest = hist_latest.drop_duplicates(subset='Parcel', keep='first')

print(f'Total history rows      : {len(hist):,}')
print(f'Unique parcels          : {len(hist_latest):,}')
print(f'Most recent year range  : {hist_latest["Hist_Tax_Year"].min():.0f} - {hist_latest["Hist_Tax_Year"].max():.0f}')
print()
print('Just_Value stats (most recent year per parcel):')
print(f'  Null/NaN  : {hist_latest["Just_Value"].isnull().sum():,}')
print(f'  Zero      : {(hist_latest["Just_Value"]==0).sum():,}')
print(f'  < $1,000  : {(hist_latest["Just_Value"]<1000).sum():,}  (will be dropped by cleaner Step 2)')
print(f'  >= $1,000 : {(hist_latest["Just_Value"]>=1000).sum():,}  (will pass cleaner)')
print(f'  Min       : ${hist_latest["Just_Value"].min():,.0f}')
print(f'  Max       : ${hist_latest["Just_Value"].max():,.0f}')
print(f'  Mean      : ${hist_latest["Just_Value"].mean():,.0f}')
print(f'  Median    : ${hist_latest["Just_Value"].median():,.0f}')

Total history rows      : 1,257,542
Unique parcels          : 111,852
Most recent year range  : 2018 - 2025

Just_Value stats (most recent year per parcel):
  Null/NaN  : 0
  Zero      : 2,490
  < $1,000  : 5,426  (will be dropped by cleaner Step 2)
  >= $1,000 : 106,426  (will pass cleaner)
  Min       : $0
  Max       : $1,079,589,000
  Mean      : $382,741
  Median    : $212,030


---
## SECTION 4 — Improvements.txt investigation

In [7]:
# Cell 6 — Year built quality check
# Pipeline imputes zero ACT_YR_BLT in cleaner Step 5

imprv['Actual_YrBlt'] = pd.to_numeric(imprv['Actual_YrBlt'], errors='coerce')
imprv['Effective_YrBlt'] = pd.to_numeric(imprv['Effective_YrBlt'], errors='coerce')
imprv['Heated_SquareFeet'] = pd.to_numeric(imprv['Heated_SquareFeet'], errors='coerce')

# Exclude SOHM/NSOHM rows (same as load_cama.py)
imprv_clean = imprv[~imprv['Imprv_Type'].astype(str).str.strip().isin(['SOHM','NSOHM'])]

print(f'Total improvement rows  : {len(imprv):,}')
print(f'After SOHM/NSOHM filter : {len(imprv_clean):,}')
print()
print('Actual_YrBlt:')
print(f'  Zero/null   : {(imprv_clean["Actual_YrBlt"]<=0).sum():,}  (will be imputed by cleaner)')
print(f'  Valid range  : {imprv_clean[imprv_clean["Actual_YrBlt"]>0]["Actual_YrBlt"].min():.0f} - {imprv_clean[imprv_clean["Actual_YrBlt"]>0]["Actual_YrBlt"].max():.0f}')
print()
print('Heated_SquareFeet:')
print(f'  Zero        : {(imprv_clean["Heated_SquareFeet"]==0).sum():,}')
print(f'  Valid (>0)  : {(imprv_clean["Heated_SquareFeet"]>0).sum():,}')
print(f'  Mean        : {imprv_clean["Heated_SquareFeet"].mean():.0f} sqft')
print(f'  Median      : {imprv_clean["Heated_SquareFeet"].median():.0f} sqft')
print()
print('Imprv_Type distribution:')
print(imprv['Imprv_Type'].value_counts().head(10))

Total improvement rows  : 179,951
After SOHM/NSOHM filter : 98,530

Actual_YrBlt:
  Zero/null   : 1  (will be imputed by cleaner)
  Valid range  : 1012 - 2026

Heated_SquareFeet:
  Zero        : 16
  Valid (>0)  : 98,513
  Mean        : 5510 sqft
  Median      : 1635 sqft

Imprv_Type distribution:
Imprv_Type
SOHM          80082
0100          64977
0400           7309
0800           6583
0300           4522
2600           2964
NSOHM          1339
4900           1262
2700           1203
8400            943
Name: count, dtype: int64


---
## SECTION 5 — Sales.txt investigation (CRITICAL — QUAL_CD1 bug)

In [8]:
# Cell 7 — QUAL_CD1 value audit
# THIS IS THE KEY BUG:
# API data had single-letter codes: 'U' = unqualified
# ZIP data has full descriptions: 'Q-QUALIFIED', '11-Corrective Deed' etc.
# features.py Step 10 does: == 'U'  which will NEVER match ZIP data
# Fix needed: check if value STARTS WITH 'U' instead of equals 'U'

sales['Sale_Price'] = pd.to_numeric(sales['Sale_Price'], errors='coerce')
sales['Sale_Line_Num'] = pd.to_numeric(sales['Sale_Line_Num'], errors='coerce')

print('ALL DOR_Qual_Code values in Sales.txt:')
print(sales['DOR_Qual_Code'].value_counts())
print()

# Show what the API code check would find (wrong)
old_check = (sales['DOR_Qual_Code'].astype(str).str.strip().str.upper() == 'U').sum()
print(f'Old check (== U):      finds {old_check} records  ← BROKEN for ZIP data')

# Show what the fixed check finds (correct)
new_check = sales['DOR_Qual_Code'].astype(str).str.strip().str.upper().str.startswith('U').sum()
print(f'Fixed check (startswith U): finds {new_check} records  ← CORRECT for ZIP data')
print()
print('Unqualified sale codes found (startswith U):')
print(sales[sales['DOR_Qual_Code'].astype(str).str.strip().str.upper().str.startswith('U')]['DOR_Qual_Code'].value_counts())

ALL DOR_Qual_Code values in Sales.txt:
DOR_Qual_Code
U-UNQUALIFIED                        174866
Q-QUALIFIED                          134511
11-Corrective Deed                    82976
01-Qualified Examina of Deed          66584
05-Qualified, Multi Trans             15156
12-Financial Inst Deed                 7705
30-Affiliated Parties                  5192
02-Qualified Credible Eviden           4360
37-Not Exp/Open Mkt-Not Infm           3477
18-Government Agency                   1711
03-Qualified, Physical Chg             1661
16-Partial Inter Deed <100in           1522
98-Unresolved Deed Errors              1314
38-Ext Circm, Prevent Forecl            492
17-Religious/Charitable Ent.            394
19-Bankruptcy/Personal Repr             282
43-Allocated $,Part Pkg-Bulk            253
40-Non-Market Finance/Lease             148
04-Qualified, legal chg                 129
39-Consd Diff than Doc Stamp             67
35-Non-typical Personal Prop             62
99-Sale w/in 90 days   

In [9]:
# Cell 8 — Sale price distribution
sale1 = sales[sales['Sale_Line_Num'] == 0].copy()
sale1['Sale_Price'] = pd.to_numeric(sale1['Sale_Price'], errors='coerce')

print(f'Most recent sales (line 0): {len(sale1):,} records')
print(f'Zero price (never sold)   : {(sale1["Sale_Price"]==0).sum():,}')
print(f'Has sale price            : {(sale1["Sale_Price"]>0).sum():,}')
print(f'Mean sale price           : ${sale1["Sale_Price"].mean():,.0f}')
print(f'Median sale price         : ${sale1["Sale_Price"].median():,.0f}')

Most recent sales (line 0): 110,180 records
Zero price (never sold)   : 5,401
Has sale price            : 104,749
Mean sale price           : $364,588
Median sale price         : $76,500


---
## SECTION 6 — Full join simulation (what load_cama.py will produce)

In [10]:
# Cell 9 — Simulate the join and check parcel coverage
# This shows how many parcels survive the full join

prop_parcels  = set(prop['Parcel'].dropna())
own_parcels   = set(own['Parcel'].dropna())
hist_parcels  = set(hist['Parcel'].dropna())
land_parcels  = set(land['Parcel'].dropna())
imprv_parcels = set(imprv['Parcel'].dropna())
sales_parcels = set(sales['Parcel'].dropna())

print(f'Property parcels  (base): {len(prop_parcels):,}')
print(f'Owners match      : {len(prop_parcels & own_parcels):,}  ({len(prop_parcels & own_parcels)/len(prop_parcels)*100:.1f}%)')
print(f'HistoryRE match   : {len(prop_parcels & hist_parcels):,}  ({len(prop_parcels & hist_parcels)/len(prop_parcels)*100:.1f}%)')
print(f'Land match        : {len(prop_parcels & land_parcels):,}  ({len(prop_parcels & land_parcels)/len(prop_parcels)*100:.1f}%)')
print(f'Improvements match: {len(prop_parcels & imprv_parcels):,}  ({len(prop_parcels & imprv_parcels)/len(prop_parcels)*100:.1f}%)')
print(f'Sales match       : {len(prop_parcels & sales_parcels):,}  ({len(prop_parcels & sales_parcels)/len(prop_parcels)*100:.1f}%)')
print()
print('Note: load_cama.py uses LEFT JOIN on Property — parcels with no sale')
print('or no improvements get NaN filled with 0. This is expected behaviour.')

Property parcels  (base): 109,360
Owners match      : 109,359  (100.0%)
HistoryRE match   : 108,255  (99.0%)
Land match        : 102,054  (93.3%)
Improvements match: 91,499  (83.7%)
Sales match       : 105,495  (96.5%)

Note: load_cama.py uses LEFT JOIN on Property — parcels with no sale
or no improvements get NaN filled with 0. This is expected behaviour.


In [11]:
# Cell 10 — Residential parcel estimate after full pipeline
# Simulates: load → cleaner → how many records survive?

prop['DOR_norm'] = prop['Prop_Use_Code'].astype(str).str.strip().str[:3]
res_parcels = prop[prop['DOR_norm'].isin(RESIDENTIAL)]

# JV filter simulation
hist['Just_Value_num'] = pd.to_numeric(hist['Just_Value'], errors='coerce')
hist_latest2 = hist.sort_values('Hist_Tax_Year', ascending=False).drop_duplicates('Parcel', keep='first')
valid_jv = set(hist_latest2[hist_latest2['Just_Value_num'] >= 1000]['Parcel'])

after_res  = len(res_parcels)
after_jv   = len(res_parcels[res_parcels['Parcel'].isin(valid_jv)])

print('ESTIMATED PIPELINE SURVIVAL:')
print(f'  All parcels in ZIP        : {len(prop):,}')
print(f'  After residential filter  : {after_res:,}  (cleaner Step 1)')
print(f'  After JV >= $1000 filter  : {after_jv:,}  (cleaner Step 2)')
print(f'  IQR outlier removal       : ~few hundred more removed (cleaner Step 3)')
print(f'  Estimated final records   : ~{int(after_jv * 0.97):,}')
print()
print(f'Your TOTAL_GOAL = 8000  →  ', end='')
if after_jv >= 8000:
    print(f'ZIP has enough data ✓ (has {after_jv:,} residential records)')
else:
    print(f'ZIP only has {after_jv:,} residential records — reduce TOTAL_GOAL')

ESTIMATED PIPELINE SURVIVAL:
  All parcels in ZIP        : 109,360
  After residential filter  : 79,594  (cleaner Step 1)
  After JV >= $1000 filter  : 79,546  (cleaner Step 2)
  IQR outlier removal       : ~few hundred more removed (cleaner Step 3)
  Estimated final records   : ~77,159

Your TOTAL_GOAL = 8000  →  ZIP has enough data ✓ (has 79,546 residential records)


---
## SECTION 7 — The fix needed in features.py

In [12]:
# Cell 11 — Show exactly what line needs to change in features.py

print('='*60)
print('FIX REQUIRED IN features.py  Step 10')
print('='*60)
print()
print('CURRENT CODE (line ~200 in features.py):')
print('  == U')
print()
print('REPLACE WITH:')
print('  .str.startswith("U")')
print()
print('WHY:')
print('  API data used single-letter codes  : U, Q, etc.')
print('  ZIP data uses full descriptions    : U-UNQUALIFIED, Q-QUALIFIED')
print('  startswith("U") works for BOTH sources')
print()
print('EXACT LINE TO CHANGE in _step10_distressed_sale_history():')
print()
print('  BEFORE:')
print('    == "U"')
print()
print('  AFTER:')
print('    .str.startswith("U")')

FIX REQUIRED IN features.py  Step 10

CURRENT CODE (line ~200 in features.py):
  == U

REPLACE WITH:
  .str.startswith("U")

WHY:
  API data used single-letter codes  : U, Q, etc.
  ZIP data uses full descriptions    : U-UNQUALIFIED, Q-QUALIFIED
  startswith("U") works for BOTH sources

EXACT LINE TO CHANGE in _step10_distressed_sale_history():

  BEFORE:
    == "U"

  AFTER:
    .str.startswith("U")


---
## SECTION 8 — Owners.txt investigation

In [13]:
# Cell 12 — Owner state distribution
# features.py Step 4 flags out-of-state owners

print('Owner state distribution (top 15):')
print(own['Owner_Mail_State'].value_counts().head(15))
print()

fl = (own['Owner_Mail_State'].astype(str).str.strip() == 'FL').sum()
oos = (
    (own['Owner_Mail_State'].astype(str).str.strip() != 'FL') &
    (own['Owner_Mail_State'].astype(str).str.strip() != '') &
    (own['Owner_Mail_State'].astype(str).str.strip() != 'nan')
).sum()

print(f'Florida owners     : {fl:,}  ({fl/len(own)*100:.1f}%)')
print(f'Out-of-state owners: {oos:,}  ({oos/len(own)*100:.1f}%)  ← is_out_of_state_owner feature')

Owner state distribution (top 15):
Owner_Mail_State
FL    103140
GA       812
TX       773
CA       757
NY       424
NC       281
PA       208
VA       193
UT       190
IL       168
MD       159
TN       144
IN       140
AZ       136
NJ       131
Name: count, dtype: int64

Florida owners     : 103,140  (94.3%)
Out-of-state owners: 6,151  (5.6%)  ← is_out_of_state_owner feature


In [14]:
# Cell 13 — Corporate owner check
# features.py Step 2 checks OWN_NAME for LLC, INC, CORP, TRUST etc.

CORPORATE_KEYWORDS = ['LLC','INC','CORP','TRUST','ESTATE','PROPERTIES',
                      'GROUP','HOLDINGS','PARTNERS','INVESTMENTS',
                      'REALTY','ENTERPRISES','LTD']

corp_mask = own['Owner_Mail_Name'].astype(str).str.upper().str.contains(
    '|'.join(CORPORATE_KEYWORDS), na=False
)
print(f'Corporate owners  : {corp_mask.sum():,}  ({corp_mask.mean()*100:.1f}%)')
print(f'Individual owners : {(~corp_mask).sum():,}  ({(~corp_mask).mean()*100:.1f}%)')
print()
print('Sample corporate owner names:')
print(own[corp_mask]['Owner_Mail_Name'].head(10).tolist())

Corporate owners  : 32,715  (29.9%)
Individual owners : 76,661  (70.1%)

Sample corporate owner names:
['ARLINGTON SQUARE/WISTERIA DOWNS LIMITED PARTNERSHIP', 'ARLINGTON SQUARE/WISTERIA DOWNS LIMITED PARTNERSHIP', 'ARMSTRONG SR & ARMSTRONG TRUSTEES', 'AUTREY GLEN TRUSTEE', 'ARTHUR ANN M LIFE ESTATE', 'AUSTIN JERRY G LIFE ESTATE', 'JONES GWENETTE M LIFE ESTATE', 'ARES THOMAS MICHAEL & DIANNA JUDD LIFE ESTATE', 'ASHLEY MOOCHELE LLC', 'ASHLEY MOOCHELE LLC']


---
## SECTION 9 — Final readiness summary

In [15]:
# Cell 14 — Final summary before running load_cama.py

print('='*60)
print('FINAL READINESS CHECK')
print('='*60)
checks = [
    ('Property.txt columns match',      True),
    ('Owners.txt columns match',        True),
    ('HistoryRE.txt columns match',     True),
    ('Land.txt columns match',          True),
    ('Improvements.txt columns match',  True),
    ('Sales.txt columns match',         True),
    ('Enough residential records',      after_res >= 8000),
    ('DEMO_MODE = False in config',     True),
    ('QUAL_CD1 fix needed in features', False),  # needs fix
]
for label, status in checks:
    icon = '✓' if status else '✗ FIX NEEDED'
    print(f'  {icon}  {label}')

print()
print('NEXT STEPS:')
print('  1. Fix features.py Step 10  (== U  →  startswith U)')
print('  2. Run: python load_cama.py')
print('  3. Run: python check_db.py  (verify data loaded)')
print('  4. Run: python main.py')

FINAL READINESS CHECK
  ✓  Property.txt columns match
  ✓  Owners.txt columns match
  ✓  HistoryRE.txt columns match
  ✓  Land.txt columns match
  ✓  Improvements.txt columns match
  ✓  Sales.txt columns match
  ✓  Enough residential records
  ✓  DEMO_MODE = False in config
  ✗ FIX NEEDED  QUAL_CD1 fix needed in features

NEXT STEPS:
  1. Fix features.py Step 10  (== U  →  startswith U)
  2. Run: python load_cama.py
  3. Run: python check_db.py  (verify data loaded)
  4. Run: python main.py
